In [ ]:
import os
from google.colab import drive

# 1. Drive'ı Mount Et (Eğer zaten bağlıysa 'Drive already mounted' der, sorun yok)
drive.mount('/content/drive')

# --- DÜZELTME BURADA ---
# Ekran görüntüsüne göre klasörün adı 'Bottleneck' (Büyük B)
# Yol: /content/drive/MyDrive/Bottleneck
BASE_DIR = '/content/drive/MyDrive/Bottleneck'

# Kontrol Bloğu: Dosyaların gerçekten orada olup olmadığını test edelim
print(f"Hedef Klasör: {BASE_DIR}")

if os.path.exists(BASE_DIR):
    print("✅ Klasör bulundu!")
    files = os.listdir(BASE_DIR)
    print("Klasör içindeki dosyalar:", files)

    # Zip dosyaları orada mı diye özel kontrol
    required_zips = ['FLAT_CMPL.zip', 'FLAT_RCL_PRE_2010.zip', 'FLAT_RCL_POST_2010.zip']
    missing_files = [f for f in required_zips if f not in files]

    if not missing_files:
        print("✅ Tüm ZIP dosyaları eksiksiz. Ana koda geçebilirsin.")
    else:
        print(f"❌ Şu dosyalar eksik: {missing_files}")
else:
    print("❌ Klasör hala bulunamadı. Lütfen dosya yolunu tekrar kontrol et.")

In [ ]:
# @title Düzeltilmiş Script: Başlıksız Veri Okuma ve Sütun Eşleştirme
# Bu kod, header olmayan NHTSA dosyalarındaki doğru sütunları indeks numaralarıyla çeker.

import pandas as pd
import numpy as np
import os
import time
from google.colab import drive
from deep_translator import GoogleTranslator
from tqdm.auto import tqdm

# -------------------------------------------------------------------------
# 1. AYARLAR
# -------------------------------------------------------------------------
DRIVE_PATH = '/content/drive/MyDrive/Bottleneck'
CHECKPOINT_FILE = os.path.join(DRIVE_PATH, 'ceviri_checkpoint.csv')

FILE_CMPL = 'FLAT_CMPL.zip'
FILE_RCL_PRE = 'FLAT_RCL_PRE_2010.zip'
FILE_RCL_POST = 'FLAT_RCL_POST_2010.zip'

N_SAMPLES_CMPL = 40000
N_SAMPLES_RCL = 5000

# Çeviri Ayarları
SOURCE_LANG = 'en'
TARGET_LANG = 'tr'

# -------------------------------------------------------------------------
# 2. VERİ OKUMA VE SÜTUN ATAMA FONKSİYONLARI
# -------------------------------------------------------------------------

def load_complaints_fixed(zip_path, n_samples):
    print(f"\n[INFO] {zip_path} işleniyor (Headerless Mod)...")
    try:
        # header=None: İlk satırı başlık yapma, veri olarak al.
        df = pd.read_csv(os.path.join(DRIVE_PATH, zip_path),
                         compression='zip',
                         sep='\t',       # NHTSA verisi genelde TAB ile ayrılır
                         header=None,    # ÖNEMLİ: Başlık yok
                         encoding='latin-1',
                         on_bad_lines='skip',
                         low_memory=False)

        # Analizimize göre sütun indeksleri:
        # 2: Üretici (MFR_NAME)
        # 11: Arıza Parçası (COMPDESC)
        # 19: Şikayet Metni (CDESCR)

        # Sadece bu sütunları alıyoruz
        target_indices = [2, 19, 11]

        # Sütun sayısı kontrolü (Dosya bozuksa indeks hatası almamak için)
        if df.shape[1] <= max(target_indices):
            print(f"[HATA] Beklenen sütun sayısı yok. Dosyadaki sütun sayısı: {df.shape[1]}")
            return None

        df_clean = df.iloc[:, target_indices].copy()

        # İsimleri Biz Atıyoruz
        df_clean.columns = ['Üretici', 'Şikayet_Metni', 'Arıza_Parçası']

        # Rastgele Örneklem
        if len(df_clean) > n_samples:
            df_clean = df_clean.sample(n=n_samples, random_state=42)

        df_clean['Veri_Tipi'] = 'Şikayet'

        return df_clean

    except Exception as e:
        print(f"[HATA] Şikayet dosyası okunurken hata: {e}")
        return None

def load_recalls_fixed(path_pre, path_post, n_samples):
    print(f"\n[INFO] Recall verileri işleniyor (Headerless Mod)...")
    try:
        # Recall dosyaları için standart NHTSA formatı indeksleri:
        # 2: Üretici (MAKETXT veya MFGNAME)
        # 8: Arıza Parçası (COMPNAME)
        # 15: Kök Neden (DESC_DEFECT)
        # 17: Çözüm (REMEDY)
        recall_indices = [2, 15, 17, 8]

        col_names = ['Üretici', 'Kök_Neden', 'Çözüm', 'Arıza_Parçası']

        df_list = []
        for path in [path_pre, path_post]:
            full_path = os.path.join(DRIVE_PATH, path)
            if os.path.exists(full_path):
                temp_df = pd.read_csv(full_path, compression='zip', sep='\t', header=None, encoding='latin-1', on_bad_lines='skip', low_memory=False)

                # İndekslerin varlığını kontrol et
                if temp_df.shape[1] > max(recall_indices):
                    temp_clean = temp_df.iloc[:, recall_indices].copy()
                    temp_clean.columns = col_names
                    df_list.append(temp_clean)
                else:
                    print(f"[UYARI] {path} dosyasında yeterli sütun yok.")

        if not df_list:
            return None

        df_final = pd.concat(df_list, ignore_index=True)

        # Rastgele Örneklem
        if len(df_final) > n_samples:
            df_final = df_final.sample(n=n_samples, random_state=42)

        df_final['Veri_Tipi'] = 'Çözüm'
        return df_final

    except Exception as e:
        print(f"[HATA] Recall dosyaları okunurken hata: {e}")
        return None

# -------------------------------------------------------------------------
# 3. İŞ AKIŞI
# -------------------------------------------------------------------------

# 1. Drive Bağla (Zaten bağlıysa geçer)
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Verileri Yükle ve Temizle
df_cmpl = load_complaints_fixed(FILE_CMPL, N_SAMPLES_CMPL)
df_rcl = load_recalls_fixed(FILE_RCL_PRE, FILE_RCL_POST, N_SAMPLES_RCL)

if df_cmpl is not None and df_rcl is not None:
    # İkisini Birleştir
    full_df = pd.concat([df_cmpl, df_rcl], ignore_index=True)

    # Sütun sırasını düzenle
    final_cols = ['Üretici', 'Şikayet_Metni', 'Arıza_Parçası', 'Kök_Neden', 'Çözüm', 'Veri_Tipi']
    full_df = full_df.reindex(columns=final_cols)

    # TEMİZLENMİŞ HAM VERİ ÇIKTISI (İLK 5 SATIR)
    print("\n" + "="*60)
    print("VERİ HAZIRLIĞI TAMAMLANDI - İLK 5 SATIR (Sütunlar Doğrulandı)")
    print("="*60)
    # NaN olan yerleri 'Yok' olarak gösterelim ki tablo okunsun
    print(full_df.head().fillna("-").to_markdown(index=False))

    print(f"\nToplam Veri Sayısı: {len(full_df)}")
    print("Sütunlar başarıyla izole edildi. Çeviriye geçiliyor...")

    # ---------------------------------------------------------------------
    # 4. ÇEVİRİ DÖNGÜSÜ (CHECKPOINT DESTEKLİ)
    # ---------------------------------------------------------------------

    # Checkpoint var mı?
    if os.path.exists(CHECKPOINT_FILE):
        saved_df = pd.read_csv(CHECKPOINT_FILE)
        start_idx = len(saved_df)
        print(f"\n[INFO] Checkpoint bulundu. {start_idx}. satırdan devam ediliyor.")
    else:
        start_idx = 0
        # Dosyayı oluştur ve header yaz
        pd.DataFrame(columns=final_cols).to_csv(CHECKPOINT_FILE, index=False)

    translator = GoogleTranslator(source=SOURCE_LANG, target=TARGET_LANG)

    # Sadece kalan veriyi al
    if start_idx < len(full_df):
        process_df = full_df.iloc[start_idx:]

        buffer = []
        save_counter = 0

        for i, row in tqdm(process_df.iterrows(), total=len(process_df), desc="Çeviri İlerlemesi"):
            try:
                # Satırı sözlüğe çevir (Pandas serisi üzerinde işlemden hızlıdır)
                row_dict = row.to_dict()

                # Çevrilecek alanlar
                fields_to_translate = ['Şikayet_Metni', 'Arıza_Parçası', 'Kök_Neden', 'Çözüm']

                for field in fields_to_translate:
                    text = str(row_dict.get(field, ''))
                    # Boş, NaN veya çok kısa metinleri çevirme
                    if text and text.lower() != 'nan' and text != '-' and len(text) > 3:
                        # Translate
                        translated_text = translator.translate(text)
                        row_dict[field] = translated_text
                        # time.sleep(0.1) # Gerekirse açılabilir

                buffer.append(row_dict)
                save_counter += 1

                # Her 100 satırda bir diske yaz (RAM şişmesin)
                if save_counter >= 100:
                    pd.DataFrame(buffer).to_csv(CHECKPOINT_FILE, mode='a', header=False, index=False)
                    buffer = []
                    save_counter = 0

            except Exception as e:
                print(f"Hata (Index {i}): {e}")
                time.sleep(2)

        # Kalanları yaz
        if buffer:
            pd.DataFrame(buffer).to_csv(CHECKPOINT_FILE, mode='a', header=False, index=False)

    print("\n" + "="*60)
    print("İŞLEM BİTTİ. Çevrilmiş Sonuç (İlk 5 Satır):")
    final_result = pd.read_csv(CHECKPOINT_FILE)
    print(final_result.head().to_markdown(index=False))

else:
    print("Veri yükleme başarısız olduğu için çeviriye geçilemedi.")

In [ ]:
# @title ADIM 1-C: Sütun Düzeltme ve Türkçe Kontrol Çevirisi
# Bu kod, kaymış sütunları düzeltir ve sonucu doğrulamak için 5 satırı Türkçeye çevirir.

import pandas as pd
from deep_translator import GoogleTranslator
from tqdm.auto import tqdm

# 1. SÜTUNLARI DÜZELT (Manuel Gözlem Sonucu)
# Paylaştığınız tabloya göre veriler şu sırada geliyor:
# [Üretici] - [Gereksiz Notlar] - [Kök Neden (Sorun)] - [Çözüm (Remedy)]

print("[BİLGİ] Sütunlar yeniden adlandırılıyor...")
# Mevcut df_recall hafızada ise onu kullan, yoksa checkpointten oku
if 'df_recall' not in locals():
    # Checkpoint dosyasını (eğer varsa) veya yeniden yükleme fonksiyonunu kullanabiliriz
    # Hızlıca hafızadakini varsayıyoruz, yoksa uyarı verecek.
    try:
        df_recall = pd.read_csv('/content/drive/MyDrive/Bottleneck/recall_ceviri_checkpoint.csv')
    except:
        print("Veri hafızada yok, lütfen bir önceki adımı tekrar çalıştırın.")

# Sütun isimlerini içeriğe göre düzeltiyoruz
df_recall.columns = ['Üretici', 'Gereksiz_Not', 'Kök_Neden', 'Çözüm']

# Gereksiz sütunu at ve sıralamayı düzelt
df_corrected = df_recall[['Üretici', 'Kök_Neden', 'Çözüm']].copy()

print(f"\n[DÜZELTİLDİ] Yeni Sütunlar: {df_corrected.columns.tolist()}")

# 2. TÜRKÇE ÇEVİRİ TESTİ (İLK 5 SATIR)
print("\n[İŞLEM] Doğrulama için ilk 5 satır Türkçeye çevriliyor...")

translator = GoogleTranslator(source='en', target='tr')
test_samples = df_corrected.head(5).copy()

for index, row in test_samples.iterrows():
    try:
        # Kök Neden Çevirisi
        sorun_eng = str(row['Kök_Neden'])
        if len(sorun_eng) > 3:
            test_samples.at[index, 'Kök_Neden'] = translator.translate(sorun_eng[:2000]) # Hız için limitli

        # Çözüm Çevirisi
        cozum_eng = str(row['Çözüm'])
        if len(cozum_eng) > 3:
            test_samples.at[index, 'Çözüm'] = translator.translate(cozum_eng[:2000])

    except Exception as e:
        print(f"Satır {index} çevrilirken hata: {e}")

# 3. SONUCU GÖSTER
print("\n" + "="*80)
print("TÜRKÇE ÇEVİRİ KONTROLÜ (Doğru Sütunlar)")
print("="*80)
# Markdown formatında yazdır (daha okunaklı)
print(test_samples.to_markdown(index=False))

print("\n\nEĞER YUKARIDAKİ TABLO MANTIKLIYSA:")
print("1. 'Kök_Neden' sütununda arızayı (Örn: Frenler tutmuyor)")
print("2. 'Çözüm' sütununda teknik müdahaleyi (Örn: Bayi parçayı değiştirecek)")
print("görüyorsanız veri hazırdır. Onay verin, tüm veriyi bu şekilde kaydedelim.")

In [ ]:
# @title ADIM 7 (FİNAL): 5.000 Veriyi Çevir ve Kaydet
# Bu script, tespit ettiğimiz 2. (Üretici), 19. (Kök Neden) ve 21. (Çözüm) sütunları kullanır.
# Dosyaları TAB ayracı ile okur.

import pandas as pd
from deep_translator import GoogleTranslator
from tqdm.auto import tqdm
import zipfile
import io

# İlerleme çubuğu entegrasyonu
tqdm.pandas()

# DOSYA YOLLARI
zip_path_post = '/content/drive/MyDrive/Bottleneck/FLAT_RCL_POST_2010.zip'
zip_path_pre = '/content/drive/MyDrive/Bottleneck/FLAT_RCL_PRE_2010.zip'
output_path = '/content/drive/MyDrive/Bottleneck/RECALL_VERISI_5000_TR.csv'

print("[1/4] Dosyalar okunuyor (Tab Ayracı ile)...")

def veri_oku_ve_temizle(zip_path):
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            file_name = z.namelist()[0]
            # 'latin-1' ve 'tab' (\t) ayracı kullanıyoruz.
            # on_bad_lines='skip' ile bozuk satırları atlıyoruz.
            df = pd.read_csv(z.open(file_name), sep='\t', header=None, encoding='latin-1', on_bad_lines='skip', low_memory=False)

            # Sütun sayısı kontrolü (En az 22 sütun olmalı)
            if df.shape[1] < 22:
                # Bazen eski dosyalarda sütunlar farklı olabilir, dinamik bulmaya çalışalım
                # En uzun 2 sütunu bul (Kök Neden ve Çözüm için)
                uzunluklar = df.astype(str).map(len).mean()
                en_uzun = uzunluklar.nlargest(2).index.tolist()
                col_sorun = en_uzun[0]
                col_cozum = en_uzun[1]
                col_uretici = 2 if df.shape[1] > 2 else 0
            else:
                # Tespit ettiğimiz standart yapı
                col_uretici = 2
                col_sorun = 19
                col_cozum = 21

            # Sadece gerekli sütunları al
            df_temp = df.iloc[:, [col_uretici, col_sorun, col_cozum]].copy()
            df_temp.columns = ['Üretici', 'Kök_Neden', 'Çözüm']
            return df_temp

    except Exception as e:
        print(f"   -> [HATA] {zip_path} okunurken hata: {e}")
        return pd.DataFrame()

# İki dosyayı da oku ve temizle
df_post = veri_oku_ve_temizle(zip_path_post)
df_pre = veri_oku_ve_temizle(zip_path_pre)

# Birleştir
df_all = pd.concat([df_post, df_pre], ignore_index=True)

# Boş verileri temizle
df_all = df_all.dropna(subset=['Kök_Neden', 'Çözüm'])

print(f"   -> Toplam {len(df_all)} satır veri havuzu oluşturuldu.")

# ---------------------------------------------------------
print("[2/4] Rastgele 5.000 satır seçiliyor...")

sample_size = min(5000, len(df_all))
df_sample = df_all.sample(n=sample_size, random_state=42).copy()

print(f"   -> {sample_size} satır seçildi ve çeviriye hazırlanıyor.")

# ---------------------------------------------------------
print("[3/4] Çeviri işlemi başlıyor (Yaklaşık 5-10 dk)...")

translator = GoogleTranslator(source='en', target='tr')

def guvenli_ceviri(text):
    text = str(text).strip()
    # Eğer metin boşsa veya çok kısaysa çevirme
    if len(text) < 3 or text.lower() == 'nan':
        return text
    try:
        # 4000 karakter limiti (Google Translate hatasını önlemek için)
        return translator.translate(text[:4000])
    except:
        return text

# Çevirileri yap (Progress Bar ile)
print("   -> 'Kök Neden' sütunu çevriliyor...")
df_sample['Kök_Neden_TR'] = df_sample['Kök_Neden'].progress_apply(guvenli_ceviri)

print("   -> 'Çözüm' sütunu çevriliyor...")
df_sample['Çözüm_TR'] = df_sample['Çözüm'].progress_apply(guvenli_ceviri)

# ---------------------------------------------------------
print("[4/4] Dosya kaydediliyor...")

# Nihai tabloyu oluştur: Üretici, Kök Neden (TR), Çözüm (TR)
df_final = df_sample[['Üretici', 'Kök_Neden_TR', 'Çözüm_TR']]

# Kaydet
df_final.to_csv(output_path, index=False, encoding='utf-8-sig')

print("="*60)
print(f"BAŞARILI! Dosyanız hazır:\n{output_path}")
print("="*60)
print(df_final.head().to_markdown(index=False))

In [ ]:
# @title ADIM 1: Kütüphaneleri Kur ve Drive'ı Bağla
# Model eğitimi için gerekli paketleri yüklüyoruz.

print("[BAŞLATILIYOR] Kütüphaneler kuruluyor...")
!pip install transformers datasets scikit-learn accelerate -q

from google.colab import drive
import os

# Drive'ı bağla
if not os.path.exists('/content/drive'):
    print("[BAĞLANTI] Google Drive bağlanıyor...")
    drive.mount('/content/drive')
else:
    print("[BİLGİ] Drive zaten bağlı.")

print("\nHER ŞEY HAZIR! Eğitime geçebiliriz.")

In [ ]:
# @title 🎓 Akademik Kök Neden Sınıflandırma Sistemi (BERT)
# Gerekli kütüphaneler
!pip install transformers datasets evaluate accelerate scikit-learn seaborn matplotlib --quiet

import pandas as pd
import numpy as np
import torch
import seaborn as sns
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
import warnings

# Ayarlar
warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Cihaz: {device.upper()}")

# ---------------------------------------------------------
# 1. VERİ YÜKLEME VE HAZIRLIK
# ---------------------------------------------------------
file_path = '/content/drive/MyDrive/Bottleneck/RECALL_VERISI_5000_TR.csv'

try:
    df = pd.read_csv(file_path)
    df = df.dropna(subset=['Kök_Neden_TR']).reset_index(drop=True)
    print(f"✅ Veri Seti Yüklendi: {len(df)} satır")
except Exception as e:
    print(f"❌ Hata: {e}")
    raise e

# --- ETİKETLEME (LABELING) ---
# Makalenizde: "Veri seti, uzman görüşüne dayalı anahtar kelime haritalaması ile 8 ana sınıfa ayrılmıştır." diyebilirsiniz.
def categorize_issue(text):
    text = str(text).lower()
    if any(x in text for x in ['fren', 'brake', 'abs', 'hidrolik', 'pedal', 'kaliper']):
        return 'Fren Sistemi'
    elif any(x in text for x in ['motor', 'engine', 'yakıt', 'gaz', 'emisyon', 'soğutma', 'şanzıman', 'egzoz']):
        return 'Motor ve Güç Aktarma'
    elif any(x in text for x in ['yazılım', 'software', 'ekran', 'gösterge', 'bilgisayar', 'sensör', 'elektrik', 'akü', 'kablo']):
        return 'Elektrik ve Yazılım'
    elif any(x in text for x in ['lastik', 'tekerlek', 'jant', 'tire', 'sibop']):
        return 'Lastik ve Jant'
    elif any(x in text for x in ['hava yastığı', 'airbag', 'kemer', 'emniyet', 'tokalar']):
        return 'Güvenlik Ekipmanları'
    elif any(x in text for x in ['direksiyon', 'rot', 'aks', 'steering', 'süspansiyon', 'yay']):
        return 'Direksiyon ve Süspansiyon'
    elif any(x in text for x in ['kapı', 'cam', 'gövde', 'şasi', 'kilit', 'ayna', 'silecek']):
        return 'Gövde ve Donanım'
    else:
        return 'Diğer / Genel'

df['Sistem_Etiketi'] = df['Kök_Neden_TR'].apply(categorize_issue)

# Etiketleri Sayısala Çevirme
label_list = sorted(list(df['Sistem_Etiketi'].unique()))
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}
df['label'] = df['Sistem_Etiketi'].map(label2id)

print("\n📊 Sınıf Dağılımı (Makale İçin Önemli):")
print(df['Sistem_Etiketi'].value_counts())

# Dataset Bölme (%80 Eğitim, %20 Test)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
hf_data = DatasetDict({
    'train': Dataset.from_pandas(train_df),
    'test': Dataset.from_pandas(test_df)
})

# ---------------------------------------------------------
# 2. MODEL EĞİTİMİ (BERT)
# ---------------------------------------------------------
# Türkçe için en iyi BERT modellerinden biri
model_ckpt = "dbmdz/bert-base-turkish-cased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def preprocess_function(examples):
    return tokenizer(examples["Kök_Neden_TR"], truncation=True, padding="max_length", max_length=128)

tokenized_data = hf_data.map(preprocess_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt, num_labels=len(label_list), id2label=id2label, label2id=label2id
).to(device)

# Akademik Metrikler Fonksiyonu
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')

    return {"accuracy": acc, "f1": f1}

args = TrainingArguments(
    output_dir="bert-classifier-academic",
    learning_rate=2e-5,
    per_device_train_batch_size=32, # A100/T4 için uygun
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True, # Hızlandırma
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("\n🚀 Eğitim Başlıyor...")
trainer.train()

# ---------------------------------------------------------
# 3. AKADEMİK RAPORLAMA VE GÖRSELLEŞTİRME
# ---------------------------------------------------------
print("\n--- 🧪 Test Sonuçları Hesaplanıyor ---")
predictions = trainer.predict(tokenized_data["test"])
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

# 1. Confusion Matrix (Görsel)
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_list, yticklabels=label_list)
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek')
plt.title('Confusion Matrix (Kök Neden Sınıflandırması)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 2. Classification Report (Metin)
print("\n--- 📄 Sınıflandırma Raporu (Makaleye Eklenecek Tablo) ---")
print(classification_report(labels, preds, target_names=label_list, digits=4))

# ---------------------------------------------------------
# 4. MODELİ KAYDETME
# ---------------------------------------------------------
save_path = '/content/drive/MyDrive/Bottleneck/Kayitli_Modeller/BERT_Final_Classifier'
if not os.path.exists(save_path): os.makedirs(save_path)

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"\n✅ Model başarıyla kaydedildi: {save_path}")

# ---------------------------------------------------------
# 5. KÖR TEST (Canlı Deneme)
# ---------------------------------------------------------
print("\n--- 🔎 Canlı Deneme ---")
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0 if device=="cuda" else -1)
sample_text = "Araç ani fren yapıldığında sağa çekiyor ve ABS ışığı yanıyor."
result = pipe(sample_text)
print(f"Girdi: {sample_text}")
print(f"Tahmin: {result[0]['label']} (Güven: {result[0]['score']:.4f})")

In [ ]:
# @title 🧠 Hata Düzeltildi: Kayıtlı Modeli Yükle ve 100 Örnekle Test Et
# Gerekli kütüphaneler
!pip install transformers torch pandas scikit-learn --quiet

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import os
from google.colab import drive

# 1. DRIVE BAĞLANTISI
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. AYARLAR
model_path = '/content/drive/MyDrive/Bottleneck/Kayitli_Modeller/BERT_Final_Classifier'
data_path = '/content/drive/MyDrive/Bottleneck/RECALL_VERISI_5000_TR.csv'

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Cihaz: {device.upper()}")

# 3. MODELİ YÜKLEME
if os.path.exists(model_path):
    print(f"📂 Model yükleniyor: {model_path}...")
    try:
        # Tokenizer'ı yüklerken de ayarları belirtelim
        loaded_tokenizer = AutoTokenizer.from_pretrained(model_path, model_max_length=512)
        loaded_model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)

        # Pipeline oluştururken tokenizer'ı açıkça veriyoruz
        classifier = pipeline("text-classification", model=loaded_model, tokenizer=loaded_tokenizer, device=0 if device=="cuda" else -1)
        print("✅ Model başarıyla yüklendi!")
    except Exception as e:
        print(f"❌ Model yüklenirken hata: {e}")
        raise e
else:
    raise FileNotFoundError("Model klasörü bulunamadı.")

# 4. 100 ÖRNEK SEÇİMİ
print("\n📊 Veri setinden 100 rastgele örnek seçiliyor...")
df = pd.read_csv(data_path)
df = df.dropna(subset=['Kök_Neden_TR'])

# Etiketleme Fonksiyonu (Kıyaslama için)
def categorize_issue(text):
    text = str(text).lower()
    if any(x in text for x in ['fren', 'brake', 'abs', 'hidrolik']): return 'Fren Sistemi'
    if any(x in text for x in ['motor', 'engine', 'yakıt', 'gaz']): return 'Motor ve Güç Aktarma'
    if any(x in text for x in ['yazılım', 'elektrik', 'sensör']): return 'Elektrik ve Yazılım'
    if any(x in text for x in ['lastik', 'jant', 'tekerlek']): return 'Lastik ve Jant'
    if any(x in text for x in ['hava yastığı', 'kemer']): return 'Güvenlik Ekipmanları'
    if any(x in text for x in ['direksiyon', 'rot', 'aks']): return 'Direksiyon ve Süspansiyon'
    if any(x in text for x in ['kapı', 'gövde', 'şasi']): return 'Gövde ve Donanım'
    return 'Diğer / Genel'

sample_df = df.sample(n=100, random_state=42).reset_index(drop=True)
sample_df['Gerçek_Etiket'] = sample_df['Kök_Neden_TR'].apply(categorize_issue)

# 5. TAHMİN YAPMA (DÜZELTME BURADA)
print("🚀 Tahminler yapılıyor...")

predictions = []
confidences = []

# --- HATA ÇÖZÜMÜ: truncation=True ve max_length=512 EKLENDİ ---
# Bu sayede 512 kelimeden uzun şikayetler otomatik kesilir ve hata vermez.
results = classifier(
    sample_df['Kök_Neden_TR'].tolist(),
    batch_size=16,
    truncation=True,
    max_length=512
)

for res in results:
    predictions.append(res['label'])
    confidences.append(res['score'])

sample_df['Model_Tahmini'] = predictions
sample_df['Guven_Skoru'] = confidences

# 6. SONUÇLARI GÖSTERME
sample_df['Durum'] = sample_df.apply(lambda x: '✅ DOĞRU' if x['Gerçek_Etiket'] == x['Model_Tahmini'] else '❌ YANLIŞ', axis=1)

accuracy = (sample_df['Durum'] == '✅ DOĞRU').mean() * 100
print(f"\n🏆 100 Örnek Üzerindeki Başarı: %{accuracy:.2f}")

print("\n--- ⚠️ Hata Analizi (Modelin Yanıldığı Durumlar) ---")
errors = sample_df[sample_df['Durum'] == '❌ YANLIŞ']
if not errors.empty:
    for idx, row in errors.head(5).iterrows():
        print(f"Sorun: {row['Kök_Neden_TR'][:100]}...")
        print(f"Gerçek: {row['Gerçek_Etiket']} | Tahmin: {row['Model_Tahmini']} (Güven: {row['Guven_Skoru']:.2f})")
        print("-" * 50)
else:
    print("Tebrikler! Model hiç hata yapmadı.")

# İlk 10 Sonuç
print("\n--- 📋 İlk 10 Tahmin ---")
print(sample_df[['Kök_Neden_TR', 'Gerçek_Etiket', 'Model_Tahmini', 'Durum']].head(10).to_markdown(index=False))

output_path = '/content/drive/MyDrive/Bottleneck/100_Ornek_Model_Test_Sonuclari.csv'
sample_df.to_csv(output_path, index=False)
print(f"\n💾 Detaylı sonuç tablosu kaydedildi: {output_path}")

In [ ]:
# @title 🎓 Akademik Performans Değerlendirme Modülü (Final Rapor)
# Gerekli kütüphaneler
!pip install transformers torch pandas scikit-learn seaborn matplotlib --quiet

import torch
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    f1_score,
    roc_curve,
    auc,
    precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize
from google.colab import drive
import os
import warnings

# Ayarlar
warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Cihaz: {device.upper()}")

# 1. BAĞLANTILAR
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

model_path = '/content/drive/MyDrive/Bottleneck/Kayitli_Modeller/BERT_Final_Classifier'
data_path = '/content/drive/MyDrive/Bottleneck/RECALL_VERISI_5000_TR.csv'

# 2. VERİ HAZIRLIĞI
print("\n📊 Veri seti ve Model yükleniyor...")
df = pd.read_csv(data_path)
df = df.dropna(subset=['Kök_Neden_TR'])

# Etiketleme Fonksiyonu (Ground Truth)
def categorize_issue(text):
    text = str(text).lower()
    if any(x in text for x in ['fren', 'brake', 'abs', 'hidrolik']): return 'Fren Sistemi'
    if any(x in text for x in ['motor', 'engine', 'yakıt', 'gaz']): return 'Motor ve Güç Aktarma'
    if any(x in text for x in ['yazılım', 'elektrik', 'sensör']): return 'Elektrik ve Yazılım'
    if any(x in text for x in ['lastik', 'jant', 'tekerlek']): return 'Lastik ve Jant'
    if any(x in text for x in ['hava yastığı', 'kemer']): return 'Güvenlik Ekipmanları'
    if any(x in text for x in ['direksiyon', 'rot', 'aks']): return 'Direksiyon ve Süspansiyon'
    if any(x in text for x in ['kapı', 'gövde', 'şasi']): return 'Gövde ve Donanım'
    return 'Diğer / Genel'

df['Sistem_Etiketi'] = df['Kök_Neden_TR'].apply(categorize_issue)
label_list = sorted(list(df['Sistem_Etiketi'].unique()))
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}
df['label'] = df['Sistem_Etiketi'].map(label2id)

# Test Seti (%20 - Eğitimde kullanılanın aynısı olması için random_state=42)
from sklearn.model_selection import train_test_split
_, test_df = train_test_split(df, test_size=0.2, random_state=42)

# HuggingFace Dataset
from datasets import Dataset
test_dataset = Dataset.from_pandas(test_df)

# 3. MODELİ YÜKLEME
tokenizer = AutoTokenizer.from_pretrained(model_path, model_max_length=512)
model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)

def preprocess_function(examples):
    return tokenizer(examples["Kök_Neden_TR"], truncation=True, padding="max_length", max_length=128)

tokenized_test = test_dataset.map(preprocess_function, batched=True)

# 4. TAHMİN (OLASILIKLARLA BERABER)
print("🚀 Test seti üzerinde tahminler alınıyor...")
trainer = Trainer(model=model)
predictions = trainer.predict(tokenized_test)
preds_probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=-1).numpy()
preds = np.argmax(preds_probs, axis=1)
labels = predictions.label_ids

# ---------------------------------------------------------
# RAPOR 1: GENEL METRİKLER
# ---------------------------------------------------------
acc = accuracy_score(labels, preds)
f1 = f1_score(labels, preds, average='weighted')
print(f"\n🏆 GENEL BAŞARI (ACCURACY): %{acc*100:.2f}")
print(f"⚖️ DENGELİ SKOR (F1-SCORE): {f1:.4f}")

# ---------------------------------------------------------
# RAPOR 2: CONFUSION MATRIX (GÖRSEL)
# ---------------------------------------------------------
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_list, yticklabels=label_list)
plt.title('Confusion Matrix (Karmaşıklık Matrisi)')
plt.ylabel('Gerçek Sınıf')
plt.xlabel('Tahmin Edilen Sınıf')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('Confusion_Matrix_Academic.png', dpi=300) # Kaydeder
plt.show()

# ---------------------------------------------------------
# RAPOR 3: ROC EĞRİSİ ve AUC (ÇOK ÖNEMLİ)
# ---------------------------------------------------------
# Çok sınıflı ROC için sınıfları binarize etmemiz lazım
y_test_bin = label_binarize(labels, classes=range(len(label_list)))
n_classes = y_test_bin.shape[1]

plt.figure(figsize=(10, 8))
colors = plt.cm.get_cmap('tab10', n_classes)

for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], preds_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors(i), lw=2,
             label=f'{label_list[i]} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (Yanlış Alarm Oranı)')
plt.ylabel('True Positive Rate (Yakakalama Oranı)')
plt.title('ROC Curves (Sınıf Bazlı Ayırt Edicilik)')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.savefig('ROC_Curve_Academic.png', dpi=300)
plt.show()

# ---------------------------------------------------------
# RAPOR 4: BIAS VE DETAYLI PERFORMANS ANALİZİ
# ---------------------------------------------------------
print("\n--- 🧐 SINIF BAZLI PERFORMANS VE BIAS ANALİZİ ---")
report = classification_report(labels, preds, target_names=label_list, output_dict=True)
bias_df = pd.DataFrame(report).transpose()

# Bias Yorumu İçin Tablo Düzenleme
bias_df = bias_df.drop(['accuracy', 'macro avg', 'weighted avg'])
bias_df = bias_df.sort_values(by='f1-score', ascending=False)

print(bias_df[['precision', 'recall', 'f1-score', 'support']].to_markdown())

print("\n🔍 BIAS YORUMU:")
en_kotu = bias_df.index[-1]
en_iyi = bias_df.index[0]
print(f"- Model '{en_iyi}' sınıfında en yüksek başarıyı gösteriyor (F1: {bias_df.loc[en_iyi, 'f1-score']:.2f}).")
print(f"- Model '{en_kotu}' sınıfında zorlanıyor (F1: {bias_df.loc[en_kotu, 'f1-score']:.2f}).")
print("  (Düşük sınıflar genellikle veri setindeki örnek azlığından kaynaklanır, buna 'Data Imbalance Bias' denir.)")

# Sonuçları CSV yapıp indirelim
bias_df.to_csv('Akademik_Sonuc_Tablosu.csv')
print("\n💾 Tüm metrikler 'Akademik_Sonuc_Tablosu.csv' dosyasına kaydedildi.")